# hunt24 主要设计思路

## 主要模块及功能

- main : 管道总入口
- workflow : 管道总导演
- db : 数据存储, 包括原始数据,最终执行结果 和 关键的中间结果 ,这个是一个基础平台类
- - schema_aligner: 数据库清洗, 与actions模块配合使用
- - actions: 配置三层数据清洗的动作
- pg_core : PriorGMM, 精筛
- - eps_esimator: CastroGinardEpsEstimator, 星团自适应参数(dbscan方法的eps)解算器（基于Castro-Ginard 论文算法实现）
- - transformer: 天文学相关的坐标转换, 单位转换 和 物理量变换等, 作为pg_core模块的输入

- validator: 对pg_core的数据,进行各种天文学和文献审计

- reportor: 报告输出

- cluster: 星团的配置, 描述星团物理特征与理论演化模型的实体对象（Data & Physical Identity）。封装了星团的中心、运动学逆协方差矩阵以及测光等龄线 DNA。
- 


以下是为您深度升级后的 **`hunt24-audit` 科学管线架构白皮书 1.3**。

在此版本中，我将 **Stage 1.5 正式确立为“面向全量大盘（百万级 Gaia DR3 原始观测数据）的全局无状态洗练层”**，并且细化了它在整个数仓资产演进中的物理角色，同时更新了 Mermaid 拓扑结构。

---

# 🌌 hunt24-audit 科学管线架构白皮书 1.3 (含多源大盘洗练)

## 1. 🎬 全管线宏观导演拓扑（Data Flow & Lifecycle）

整个管线围绕 `workflow`（总导演）展开，各计算阶段完全解耦，统一通过 `db`（基础设施底座）进行无状态的数据传递与状态持久化。

管线建立了由浅入深的三道核心“数据防御体系”：

1. **Stage 1.2（外部数据吸纳）**：通过 `db.AssetManager` 拦截磁盘外部异构文件（原始 Gaia 视场包、文献星表），依据 `config.py` 的策略原子化吸纳至数仓 Raw 层。
2. **Stage 1.5（全局大盘洗练）**：**【本版核心升级】** 专注于对 Stage 1.2 摄入的目标星团大范围原始观测数据（百万级 Gaia DR3 巨量非结构化大盘）执行语法对齐、列名映射和坏数据修剪。它是全局无状态的，其产物 `stx_view` 是一笔可供多方复用的标准化高价值数仓资产。
3. **Stage 2（空间物理预切片）**：在标准视图 `stx_view` 基础上，利用 `cluster` 对象的动态天体物理文献先验执行硬范围裁剪。将百万级数据流在毫秒内瞬间压缩至万级候选星盘，彻底防御下游 `pg_core` 计算自适应超参时的性能灾难。

```mermaid
flowchart TD
    %% 节点定义
    main["main.py (命令行入口)"]
    config["config.py (静态蓝图)"]
    workflow["workflow.py (总导演)"]
    cluster["cluster.py (动态物理实体)"]
    
    subgraph DB_Layer ["🛠️ DB 基础设施支持层"]
        asset["Stage 1.2: db.AssetManager"]
        aligner["Stage 1.5: schema_aligner"]
    end

    slice["Stage 2: workflow 空间物理切片"]
    transformer["Stage 3: transformer 变换"]
    
    subgraph PG_Core ["🔴 Stage 4: pg_core (PriorGMMEx 精筛核)"]
        eps["eps_estimator (Castro-Ginard 算法)"]
        gmm["GMM 软概率迭代 (EM 算法)"]
    end

    validator["Stage 5: validator 联合审计"]
    reportor["Stage 6: reportor 资产交付"]

    %% 依赖与数据流向
    main --> config
    config --> workflow
    workflow --> cluster
    
    cluster --> asset
    config -.->|读取数据加载与写入策略| asset
    
    asset -->|吸纳: 原始百万级 Gaia DR3 磁盘文件| aligner
    aligner -->|洗练: 对齐字段、剔除物理噪声、生成宽表视图| slice
    
    cluster -->|提取文献中心先验| slice
    slice -->|裁切: 1~3万 条安全切片内存盘| transformer
    transformer --> PG_Core
    
    eps -->|自适应唯一最优 EPS| gmm
    gmm --> validator
    validator --> reportor

```

---

## 2. ⏳ 核心数据血缘演进流向

在整个管线生命周期中，天体测量学数据由“磁盘杂乱文件”逐步提纯为“软概率科学资产”，其血缘边界不可混淆：

```mermaid
sequenceDiagram
    autonumber
    participant Ext as 外部磁盘 (CSV/VOTable)
    participant Raw as db.Raw层 (非结构化)
    participant View as db.stx_view (标准宽表)
    participant Slic as 内存.df_target_sliced (万级)
    participant Core as pg_core (GMM 先验)

    Note over Ext, Raw: Stage 1.2: 资产加载防线
    Ext->>Raw: AssetManager 原子吸纳全大盘 Gaia 数据
    
    Note over Raw, View: Stage 1.5: 楚河汉界 - 全局无状态洗练
    Raw->>View: schema_aligner 执行列名对齐、单位对齐 (对几百万星)
    
    Note over View, Slic: Stage 2: 空间物理硬防御
    View->>Slic: workflow 依据 cluster 先验中心拉框，毫秒级快速裁切
    
    Note over Slic, Core: Stage 4: 高维精筛数学域
    Slic->>Core: 输入安全的切片大盘驱动自适应超参反演与似然面收敛

```

---

## 3. 🧩 核心模块深度物理语义说明

### ⚙️ 配置与声明层（Blueprint & Configuration）

* **`config` (全局静态配置清单)**
* **职责**：整个管线的**静态配置蓝图与出厂设定（Compile-time Blueprint）**。
* **核心内容**：
1. **数据加载策略与资产清单（MANIFEST / IngestConfig）**：死锁了 Stage 1.2 外源文件的本地存储拓扑路径、数据表的覆盖/追加写策略、完整性校验机制以及外部数据导入数仓的底层映射规范。
2. **天体测量学基准（CatalogConfig）**：声明了多源文献异构字段（如 Gaia DR3、指定比对星表）的原始映射字典、单位转换因子（如角度转弧度、视差转距离）及硬断点参数。
3. **算法超参白皮书**：死锁了 `min_pts`（9星）、KDE 随机重采样迭代次数（30次）等不随运行状态改变的科学常数。





### 🎬 调度与标识层（Orchestration & Identity）

* **`main` (管道总入口)**
* **职责**：整个系统的**命令行生命周期触发器**。负责解析用户输入的命令行参数（目标星团 `-c`、对比星表 `-cat`、特征计算模式 `-m`、初筛算法 `-a`、产出等级 `--result` 以及装载策略 `-p` 等），配置全局日志级别，并在激活 `-mt` 时挂起常规计算链、直通数仓底座执行索引与备份维护。


* **`workflow` (管道总导演)**
* **职责**：**宏观生命周期总调度器（Orchestrator）**。本身不参与底层的任何具体数学计算，负责控制管线各阶段（Stage 1 ~ Stage 6）的自然演进，将上游的计算产物以 DataFrame 级别的数据流优雅地喂给下游算子。


* **`cluster` (星团动态物理实体)**
* **职责**：星团的 **Data & Physical Identity（数字与物理真身）**。
* **机制**：它是一个强类型的动态领域对象。初始化时通过绑定的数仓底座，并根据 `-p` 参数的装载策略（`static` / `dynamic`），自动吸纳或自适应同步该星团在科学文献中的核心先验参数（如 $\alpha, \delta, \varpi, \mu_{\alpha*}, \mu_{\delta}$ 中心、逆协方差矩阵以及测光等龄线 DNA 矩阵），为管线演进提供全局物理上下文。



### 🗄️ 基础设施与数仓层（Data Storage & ETL Engine）

* **`db` (数据存储底座 - 基础平台类)**
* **职责**：整个流水线的**无状态基础设施底座与科学数仓（Data Warehouse & State Persistence）**。基于高性能列式存储引擎构建。内部包含两个核心子算子：
1. **`db.AssetManager` (外部数据加载资产管理器)**：**[外源载入防线]** 专门负责对接外部磁盘文件系统。它无状态地解析并严格遵循 `config.py` 中定义的加载策略与路径注册表，支持将大规模外部 `CSV`、`VOTable` 等异构天文星表格式一键原子化吸纳（Ingest）到数仓的 Raw 层表中。具备表存在性校验、文件完整性审计、以及加载策略指定的断点覆盖/跳过机制。
2. **数仓核心引擎**：负责高效存储千万级全天区原始天体测量快照，并响应 `-mt` 维护指令。通过 actions 级联机制，在底层物理存储上明确划分、隔离出清洗层（STD）、扩展层（STX）和联合对齐层（ALN），落地中间产物并提供断点续算支持。




* **`schema_aligner` 与 `actions` (结构规范清洗引擎)**
* **职责**：**面向百万级 Gaia 大盘的无状态洗练路由。** 与 `actions` 模块配合作为 Stage 1.5 的核心，专门处理由 `AssetManager` 加载进来的原始非结构化 Gaia 大盘。它执行语法和结构层面的统一对齐（将 `parallax`, `pmra` 等标准化，剔除无视差、无自行坐标的物理死星噪音），屏蔽多源文献异构命名的噪音，在 `db` 中对齐生成统一的标准宽表视图 `stx_view`。



### 🔴 核心科学计算核（Statistical Clustering & Kinematics）

* **`pg_core` (PriorGMM / 高维精筛引擎)**
* **职责**：高斯混合模型（GMM）多维软概率联合拟合的核心计算域。它动态响应 `-m` 传入的特征维度计算模式（如 `3d`, `5d`, `6d_p` 等），计算出每颗恒星属于“该星团成分”的无偏成员概率。
* **`eps_estimator` (Castro-Ginard 超参解算器)**：**隐藏并合并在 `pg_core` 内部。** 专门用于实现 Castro-Ginard et al. (2018) 论文算法，动态响应 `-a` 算法策略。当检测到 `dbscan` 时，利用 $k$-NND 的特征突变谷值，结合高斯核密度估计（KDE）重采样，反演当前局部天区非结构化噪声的本征分界线，计算出唯一的自适应最优领域半径 `eps`。
* **`transformer` (天文学特征变换算子)**：`pg_core` 的输入特征工程引擎。负责执行高维空间矩阵标准化（消除度、mas/yr、mas 之间的标度残差）、天球坐标向银道坐标转换、视差向绝对距离变换、以及本征三维速度空间（$U, V, W$）的运动学物理反演。



### 🟡 审计与资产交付层（Verification & Delivery）

* **`validator` (天文学与文献联合审计核)**
* **职责**：对精筛收敛后的成员概率数据执行“物理 + 文献”的双路交叉审计。利用赫罗图（CMD）落线及等龄线弥散度进行天文学本征审计；同时增量调取远程/本地文献数据库，与通过 `-cat` 传入的指定参考星表类别进行异构多路分流交叉对齐审计。


* **`reportor` (资产交付与报告输出)**
* **职责**：流水线的资产交付终点。将最终确认的高置信度成员星表持久化入库 `db`，并根据 `--result` 参数级别（`brief` 级仅输出轻量可视化简报与状态日志；`detailed` 级则额外导出全量物理资产 CSV 星表），完成该星团的科学归档。



---

## 4. ⚖️ 架构防线与血缘隔离原则

为了确保千万级高吞吐量管线的健壮性，开发与重构时必须严守以下边界：

1. **静态与动态分离 (`config` vs `cluster`)**：全局不变的常量、**Stage 1.2 的资产数据加载策略与路径映射**必须死锁在 `config.py` 中；随运行星团实体动态改变的文献中心、观测特征则必须通过 `cluster.py` 对象进行动态参数同步，严禁混淆。
2. **清洗与裁剪必须“楚河汉界” (`schema_aligner` vs `workflow`)**：`schema_aligner` 属于全局大盘的清洗洗练（针对几百万星体无状态对齐），严禁在这里引入特定星团的文献先验；特定星团的相空间降维和防御拉框，必须留在 `workflow` 调度的 Stage 2 物理预切片层，从而保证 `stx_view` 资产的全局高度可复用性。
3. **高内聚低耦合（DBSCAN 不出精筛门）**：自适应 `eps` 计算及无监督聚类唯一的物理目的，是为精筛 GMM 计算核心成分的先验中心（Prior Initialization）。因此，`eps_estimator` 和聚类内核必须完全内聚隐藏在 `pg_core` 内部。
4. **数据大盘负样本空间防线**：为了保证 GMM 在期望最大化（EM）迭代时的数学收敛性，必须保留足够的背景负样本。上游粗筛洗出的 `df_seeds` 仅能用于引导先验参数，驱动整个似然面收敛的输入必须是包含大量背景野星的**物理预切片大盘**。

---